# Serving a Model with FastAPI

---

In this notebook, we will build a FastAPI application that loads the `iris_pipeline.joblib` we saved in the Model Persistence section and serves predictions through an HTTP endpoint.

We will cover:
- Loading a saved Pipeline at application startup.
- Creating a `/predict` endpoint that accepts features and returns predictions
- Defining Pydantic schemas for request and response
- Testing the API with Python's `requests` library and with Swagger UI
- Understanding the full request-response lifecycle.

> ⚠️ **Note:** This notebook explains and walks through the code. The actual runnable application lives in `app/main.py`. After studying this notebook, run the app from the `02_api_development/` directory with: `uvicorn app.main:app --reload`

---

## 1. The Big Picture

Here's what we're building:

```
Client (browser / Python script / mobile app)
    │
    │  POST /predict
    │  {"sepal_length": 5.1, "sepal_width": 3.5,
    │   "petal_length": 1.4, "petal_width": 0.2}
    ▼
FastAPI Server (Uvicorn)
    │
    ├─ Pydantic validates the input
    ├─ Pipeline transforms + predicts
    │
    ▼
Response
    {"prediction": "setosa", "prediction_id": 0,
     "probabilities": {"setosa": 0.97, "versicolor": 0.02, "virginica": 0.01}}
```

---

## 2. Loading the Model at Startup

A common pattern in ML APIs is to load the model **once** when the server starts, and keep it in memory for all subsequent requests. This avoids the overhead of reading from disk on every prediction.

FastAPI provides a **lifespan** mechanism for this:

In [ ]:
import joblib
from contextlib import asynccontextmanager
from fastapi import FastAPI

ml_model = {} # A dict to store the loaded model

@asynccontextmanager
async def lifespan(app: FastAPI):
    # --- Startup: runs once when the server starts ---
    ml_model["pipeline"] = joblib.load("../01_model_persistence/models/iris_pipeline.joblib")
    print("✅ Model loaded successfully!")
    yield
    # --- Shutdown: runs when the server stops ---
    ml_model.clear()
    print("🛑 Model unloaded.")
    
app = FastAPI(lifespan=lifespan)

| Concept | Explanation |
| :--- | :--- |
| `ml_model = {}` | A module-level dictionary that acts as a simple "store" for the loaded model. Accessible from any endpoint. |
| `@asynccontextmanager` | A Python pattern for code that runs before and after a block. `yield` is the dividing line: code before it runs at startup, code after it runs at shutdown. |
| `lifespan(app)` | FastAPI's lifecycle hook. Replaces the older `@app.on_event("startup")` pattern. |

The model lives in `ml_model["pipeline"]` for the entire lifetime of the server.

---

## 3. Defining the Input Schema

We need a Pydantic model that describes exactly what features our Iris Pipeline expects. Looking at our metadata from the previous section, the Pipeline expects 4 float features:

In [ ]:
from pydantic import BaseModel, Field

class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., gt=0, description="Length of the sepal in cm")
    sepal_width: float = Field(..., gt=0, description="Width of the sepal in cm")
    petal_length: float = Field(..., gt=0, description="Length of the petal in cm")
    petal_width: float = Field(..., gt=0, description="Width of the petal in cm")

| Part | Meaning |
| :--- | :--- |
| `Field(...)` | The `...` means this field is **required** (no default value). |
| `gt=0` | A **validator**: the value must be **g**reater **t**han 0. FastAPI returns a 422 error if someone sends a negative number. |
| `description=` | Appears in the Swagger UI documentation. Helps clients understand what to send. |

---

## 4. Defining the Response Schema

We also define what the API returns:

In [ ]:
class IrisPrediction(BaseModel):
    prediction: str
    prediction_id: int
    probabilities: dict[str, float]

This tells clients (and the Swagger docs) that they'll receive:
- `prediction`: The class name (e.g., `setosa`)
- `prediction_id`: The numeric class label (e.g., `0`)
- `probabilities`: A dict mapping each class to its probability

---

## 5. The Prediction Endpoint

Now we connect everything: the loaded model, the input schema, and the response schema.

In [ ]:
import numpy as np

TARGET_NAMES = ["setosa", "versicolor", "virginica"]

@app.post("/predict", response_model=IrisPrediction)
def predict(features: IrisFeatures):
    # 1. Convert Pydantic model to a NumPy array (the format our Pipeline expects)
    X = np.array([[
        features.sepal_length,
        features.sepal_width,
        features.petal_length,
        features.petal_width
    ]])
    
    # 2. Get the prediction and probabilities from the Pipeline
    pipeline = ml_model["pipeline"]
    prediction_id = pipeline.predict(X)[0]
    probabilities = pipeline.predict_proba(X)[0]
    
    # 3. Build and return the response
    return IrisPrediction(
        prediction=TARGET_NAMES[prediction_id],
        prediction_id=int(prediction_id),
        probabilities={
            name: round(float(prob), 4)
            for name, prob in zip(TARGET_NAMES, probabilities)
        }
    )

### Step-by-step Breakdown

1. **`features: IrisFeatures`**: FastAPI receives the JSON body, validates it againts `IrisFeatures`, and passes a validated Python object to the function.
2. **`np.array([[...]])`**: We convert the 4 feature values into a 2D NumPy array with shape `(1, 4)`; one sample, four features. This is what scikit-learn Pipelines expect.
3. **`pipeline.predict(X)`**: The Pipeline handles scaling internally (StandardScaler) and returns the predicted class ID.
4. **`pipeline.predict_proba(X)`**: Returns the probability distribution across all 3 classes.
5. **`return IrisPrediction(...)`**: We build the response object, and FastAPI serializes it to JSON.

---

## 6. A Health Check Endpoint

Every production API should have a simple health check endpoint. This is used by monitoring tools, load balancers, and deployment platforms to verify the server is alive:

In [ ]:
@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_loaded": "pipeline" in ml_model
    }


---

## 7. Testing the API

Once the server is running (`uvicorn app.main:app --reload`), you can test it in three ways:

### 7.1. Swagger UI (Browser)

Visit `http://127.0.0.1:8000/docs`, click on the `POST / predict` endpoint, click "Try it out", paste the JSON, and click "Execute".

### 7.2. Python `requests` Library

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={
        "sepal_length": 5.1,
        "sepal_width": 3.5,
        "petal_length": 1.4,
        "petal_width": 0.2
    }
)
print(response.json())

Expected output:
```json
{
    "prediction": "setosa",
    "prediction_id": 0,
    "probabilities": {"setosa": 0.97, "versicolor": 0.02, "virginica": 0.01}
}
```

### 7.3. cURL (Terminal)

In [ ]:
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}'


---

## 8. Summary

| Concept | Key Takeaway |
| :--- | :--- |
| **Lifespan** | Load the model once at startup using `@asynccontextmanager`. Keeps it in memory for all requests. |
| **Pydantic Input Schema** | Defines and validates the features your model expects. Rejects bad input automatically. |
| **Pydantic Response Schema** | Defines the shape of the API response. Documents it in Swagger. |
| **NumPy Conversion** | Convert Pydantic fields to a 2D NumPy array `(1, n_features)` before passing to the Pipeline. |
| **predict + predict_proba** | Return both the class label and the confidence scores for a richer response. |
| **Health Check** | A `GET /health` endpoint for monitoring and deployment verification. |

---

**Next:** [Request Validation with Pydantic](./03_request_validation_with_pydantic.ipynb) — Advanced validation, batch predictions, and error handling.
